<a href="https://colab.research.google.com/github/jheru1773/DigSkola/blob/main/End_to_End_ML_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# End-to-End Basic Machine Learning Deployment with Flask & Postman

Notebook ini dirancang untuk mentee Data Science agar memahami alur lengkap:

1. Business Understanding  
2. Data Loading & EDA  
3. Data Preprocessing  
4. Model Training  
5. Model Evaluation  
6. Model Serialization (`pickle`)  
7. Basic Deployment menggunakan Flask  
8. Public API Exposure menggunakan Ngrok  
9. API Testing menggunakan Postman  

Dataset: Human Capital Promotion Prediction



# 1. Import Libraries

Pada tahap awal, kita mengimpor seluruh library yang dibutuhkan untuk:
- data manipulation
- preprocessing
- modeling
- evaluation
- deployment


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# 2. Load Dataset

Dataset diambil langsung dari GitHub agar reproducible.


In [2]:
DATA_URL = "https://raw.githubusercontent.com/densaiko/data_science_learning/main/dataset/Human%20Capital.csv"
df = pd.read_csv(DATA_URL)
print("Shape Dataset:", df.shape)
df.head()

Shape Dataset: (54808, 13)


,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,0,49.0,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,60.0,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,50.0,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,50.0,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,73.0,0



# 3. Basic Exploratory Data Analysis (EDA)

Sebagai Data Scientist, sebelum modeling kita wajib memahami:
- tipe data
- missing values
- distribusi target
- distribusi categorical features


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54808 entries, 0 to 54807
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           54808 non-null  int64  
 1   department            54808 non-null  object 
 2   region                54808 non-null  object 
 3   education             52399 non-null  object 
 4   gender                54808 non-null  object 
 5   recruitment_channel   54808 non-null  object 
 6   no_of_trainings       54808 non-null  int64  
 7   age                   54808 non-null  int64  
 8   previous_year_rating  50684 non-null  float64
 9   length_of_service     54808 non-null  int64  
 10  awards_won            54808 non-null  int64  
 11  avg_training_score    52248 non-null  float64
 12  is_promoted           54808 non-null  int64  
dtypes: float64(2), int64(6), object(5)
memory usage: 5.4+ MB


In [4]:
df.isnull().sum().sort_values(ascending=False)

,0
previous_year_rating,4124
avg_training_score,2560
education,2409
employee_id,0
department,0
gender,0
region,0
no_of_trainings,0
recruitment_channel,0
age,0


In [5]:
df.describe(include="all")

,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,awards_won,avg_training_score,is_promoted
count,54808.000000,54808,54808,52399,54808,54808,54808.000000,54808.000000,50684.000000,54808.000000,54808.000000,52248.000000,54808.000000
unique,NaN,9,34,3,2,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,Sales & Marketing,region_2,Bachelor's,m,other,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,16840,12343,36669,38496,30446,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,39195.830627,NaN,NaN,NaN,NaN,NaN,1.253011,34.803915,3.329256,5.865512,0.023172,63.712238,0.085170
std,22586.581449,NaN,NaN,NaN,NaN,NaN,0.609264,7.660169,1.259993,4.265094,0.150450,13.521910,0.279137
min,1.000000,NaN,NaN,NaN,NaN,NaN,1.000000,20.000000,1.000000,1.000000,0.000000,39.000000,0.000000
25%,19669.750000,NaN,NaN,NaN,NaN,NaN,1.000000,29.000000,3.000000,3.000000,0.000000,51.000000,0.000000
50%,39225.500000,NaN,NaN,NaN,NaN,NaN,1.000000,33.000000,3.000000,5.000000,0.000000,60.000000,0.000000
75%,58730.500000,NaN,NaN,NaN,NaN,NaN,1.000000,39.000000,4.000000,7.000000,0.000000,77.000000,0.000000



## Target Distribution

Cek apakah dataset balanced atau imbalanced.


In [6]:
df["is_promoted"].value_counts()

,count
is_promoted,
0,50140
1,4668



Insight:
- Kelas `0` jauh lebih banyak dibanding `1`
- Ini disebut *imbalanced dataset*
- Accuracy saja tidak cukup untuk evaluasi model



# 4. Data Cleaning

Untuk sesi basic deployment, kita gunakan pendekatan sederhana:
- Menghapus missing values


In [7]:
df = df.dropna()
print("Shape setelah dropna:", df.shape)

Shape setelah dropna: (46380, 13)



# 5. Categorical Feature Inspection


In [8]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()
categorical_columns

['department', 'region', 'education', 'gender', 'recruitment_channel']

In [9]:
for col in categorical_columns:
    print("=" * 50)
    print(f"Column: {col}")
    print(df[col].value_counts().head())
    print()

Column: department
department
Sales & Marketing    13711
Operations            9259
Procurement           6639
Technology            6502
Analytics             4610
Name: count, dtype: int64

Column: region
region
region_2     10288
region_22     5171
region_7      4200
region_15     2373
region_13     2355
Name: count, dtype: int64

Column: education
education
Bachelor's          31767
Master's & above    14194
Below Secondary       419
Name: count, dtype: int64

Column: gender
gender
m    32347
f    14033
Name: count, dtype: int64

Column: recruitment_channel
recruitment_channel
other       25747
sourcing    19655
referred      978
Name: count, dtype: int64




# 6. Feature Engineering & Encoding

Machine Learning model sklearn tidak bisa menerima string langsung.

Karena itu kita encode categorical feature menjadi numeric menggunakan `LabelEncoder`.


In [10]:
df_processed = df.copy()
label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    # simpan encoder
    pickle.dump(le, open(f"label_encoder_{col}.pkl", "wb"))

print("Semua encoder berhasil disimpan.")

Semua encoder berhasil disimpan.



# 7. Feature Selection

Pisahkan:
- Feature (`X`)
- Target (`y`)


In [11]:
X = df_processed.drop(columns=["employee_id", "is_promoted"])
y = df_processed["is_promoted"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)

Shape X: (46380, 11)
Shape y: (46380,)


In [12]:
# Double check X's features names and y Series names
display(X.columns.to_list())
print("")
display(y.info())

['department',
 'region',
 'education',
 'gender',
 'recruitment_channel',
 'no_of_trainings',
 'age',
 'previous_year_rating',
 'length_of_service',
 'awards_won',
 'avg_training_score']


<class 'pandas.core.series.Series'>
Index: 46380 entries, 0 to 54807
Series name: is_promoted
Non-Null Count  Dtype
--------------  -----
46380 non-null  int64
dtypes: int64(1)
memory usage: 724.7 KB


None


# 8. Train-Test Split

Data training digunakan untuk belajar.  
Data testing digunakan untuk evaluasi generalization.


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

Train Shape: (37104, 11)
Test Shape : (9276, 11)



# 9. Model Training

Untuk pembelajaran deployment dasar, kita gunakan:
- Logistic Regression
- ringan
- cepat
- mudah dipahami


In [14]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.



# 10. Model Evaluation

Evaluasi dilakukan pada:
- training set
- testing set


In [15]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print("Training Accuracy :", train_acc)
print("Testing Accuracy  :", test_acc)

Training Accuracy : 0.9166936179387667
Testing Accuracy  : 0.9155886157826649


In [16]:
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96      8462
           1       0.69      0.07      0.13       814

    accuracy                           0.92      9276
   macro avg       0.80      0.53      0.54      9276
weighted avg       0.90      0.92      0.88      9276



In [17]:
confusion_matrix(y_test, y_pred_test)

array([[8436,   26],
       [ 757,   57]])


## Insight Evaluasi

Perhatikan:
- Accuracy
- Precision
- Recall
- F1-score

Karena dataset imbalance, recall dan F1-score penting diperhatikan.



# 11. Save Model

Model yang sudah training disimpan menjadi file `.pkl`.

Inilah file yang nanti digunakan saat deployment.


In [18]:
pickle.dump(model, open("model_logreg.pkl", "wb"))
print("Model berhasil disimpan.")

Model berhasil disimpan.



# 12. Install Deployment Dependencies

Kita menggunakan:
- Flask → membuat API
- pyngrok → expose localhost menjadi public URL


In [19]:
!pip install Flask==3.0.0 pyngrok==7.1.2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.2 MB/s eta 0:00:00



# 13. Import Deployment Libraries


In [20]:
from flask import Flask, request, jsonify
from pyngrok import ngrok


# 14. Setup Ngrok

1. Buat akun di ngrok
2. Copy auth token
3. Paste di bawah

https://dashboard.ngrok.com/get-started/your-authtoken


In [21]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
PORT = 5000


# 15. Create Flask API

Flow API:
1. menerima JSON dari user
2. convert ke DataFrame
3. preprocessing menggunakan encoder lama
4. prediksi menggunakan model
5. return hasil JSON


In [22]:
# load trained model
loaded_model = pickle.load(open("model_logreg.pkl", "rb"))

app = Flask(__name__)

@app.route("/", methods=["GET"])
def home():
    return {
        "message": "Human Capital Promotion Prediction API"
    }

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    df_new = pd.DataFrame([data])

    # preprocessing categorical columns
    for col in categorical_columns:
        encoder = pickle.load(open(f"label_encoder_{col}.pkl", "rb"))
        df_new[col] = encoder.transform(df_new[col])

    prediction = loaded_model.predict(df_new)[0]
    probability = loaded_model.predict_proba(df_new)[0][1]

    return jsonify({
        "prediction": int(prediction),
        "promotion_probability": float(round(probability, 4))
    })


# 16. Run Ngrok Tunnel

Notebook akan menghasilkan public URL.


In [25]:
ngrok.set_auth_token("3Gc86Z80ffjpCr00CbTWb4xYR4P_349R53nZWCC6Bi4eieTob")
public_url = ngrok.connect(PORT)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://spectator-huskiness-siding.ngrok-free.dev" -> "http://localhost:5000"



# 17. Run Flask Server

Jika berhasil:
- Flask server running
- endpoint bisa diakses via Postman


In [ ]:
app.run(port=PORT)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Jul/2026 10:04:26] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Jul/2026 10:04:26] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [18/Jul/2026 10:21:48] "POST / HTTP/1.1" 405 -



# 18. Testing API with Postman

## Endpoint
```python
POST /predict
```

## Example JSON Body

```json
{
    "department": "Sales & Marketing",
    "region": "region_7",
    "education": "Bachelor's",
    "gender": "m",
    "recruitment_channel": "sourcing",
    "no_of_trainings": 1,
    "age": 35,
    "previous_year_rating": 5.0,
    "length_of_service": 8,
    "KPIs_met >80%": 1,
    "awards_won?": 0,
    "avg_training_score": 85
}
```



# 19. Important Deployment Insights

## Common Beginner Mistakes

### ❌ Membuat encoder baru saat inference
Harus menggunakan encoder lama yang dipakai saat training.

### ❌ Urutan kolom berubah
Feature order harus konsisten.

### ❌ Training di API
Training hanya dilakukan offline.
API hanya untuk inference/predict.

### ❌ Tidak serialize preprocessing
Preprocessing object juga harus disimpan.



# 20. Real-World Data Scientist Workflow

## Training Environment
- Notebook / experimentation
- Feature engineering
- Evaluation

## Artifact Storage
- model.pkl
- encoders.pkl

## Deployment
- Flask / FastAPI
- Docker
- Cloud deployment

## Monitoring
- model drift
- latency
- prediction quality



# 21. Suggested Next Learning

Setelah notebook ini, mentee bisa lanjut ke:
- FastAPI
- Docker
- MLflow
- BentoML
- Airflow
- CI/CD deployment
- Cloud deployment (GCP/AWS/Azure)
- Model monitoring
